<a href="https://colab.research.google.com/github/AlessiaSerra/Clickbait-Detection/blob/main/ClickBaitDetection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
AIT and Simplicity Theory for News Engagement & Clickbait Detection
Algorithmic Information & Artificial Intelligence (Telecom Paris / IP Paris)
"""

import math
import lzma
import zlib
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from dataclasses import dataclass
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Support for LLM-based Surprisal / Causal Complexity
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


# ============================================================================
# 1. AIT & Simplicity Theory Estimators
# ============================================================================

class KolmogorovCompressor:
    """Approximates Kolmogorov complexity K(x) and K(x|y) via universal lossless compression."""

    @staticmethod
    def compress_bits(text: str, engine: str = "lzma") -> float:
        b = text.encode("utf-8")
        if engine == "lzma":
            c_bytes = lzma.compress(b, preset=9)
        elif engine == "zlib":
            c_bytes = zlib.compress(b, level=9)
        else:
            raise ValueError(f"Unknown engine: {engine}")
        return len(c_bytes) * 8.0

    @classmethod
    def ncd(cls, x: str, y: str, engine: str = "lzma") -> float:
        """Normalized Compression Distance (NCD) between x and y."""
        zx = cls.compress_bits(x, engine)
        zy = cls.compress_bits(y, engine)
        zxy = cls.compress_bits(x + " " + y, engine)

        min_z = min(zx, zy)
        max_z = max(zx, zy)
        if max_z == 0:
            return 0.0
        return max(0.0, (zxy - min_z) / max_z)


class WorldModelComplexity:
    """
    Estimates Generation Complexity C_w(s) representing the world model.
    C_w(s) = log2(1 / P_world(s)) via next-token autoregressive LLM surprisal.
    """
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if (TORCH_AVAILABLE and torch.cuda.is_available()) else "cpu"
        if TORCH_AVAILABLE:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
            self.model.eval()
        else:
            self.tokenizer = None
            self.model = None

    def compute_generation_complexity(self, text: str) -> float:
        if not TORCH_AVAILABLE or self.model is None:
            words = text.split()
            if not words:
                return 0.0
            freqs = {w: words.count(w) / len(words) for w in set(words)}
            ent = -sum(p * math.log2(p) for p in freqs.values())
            return ent * len(words)

        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        input_ids = inputs["input_ids"]

        if input_ids.shape[1] <= 1:
            return 8.0

        with torch.no_grad():
            outputs = self.model(**inputs, labels=input_ids)
            loss_nats = outputs.loss.item()
            num_tokens = input_ids.shape[1]
            total_bits = (loss_nats / math.log(2.0)) * num_tokens
            return float(total_bits)


# ============================================================================
# 2. Simplicity Theory Feature Extraction & Pure AIT Classifier
# ============================================================================

@dataclass
class AITFeatures:
    c_desc_headline: float      # C(H)
    c_world_headline: float     # C_w(H)
    unexpectedness_h: float     # U(H) = C_w(H) - C(H)
    ncd_headline_body: float    # NCD(H, B)
    c_desc_body: float          # C(B)
    ait_score: float            # (U(H) / C(H)) * NCD(H, B)


class NewsAITAnalyzer:
    def __init__(self, world_model: WorldModelComplexity):
        self.world_model = world_model
        self.compressor = KolmogorovCompressor()

    def extract_features(self, headline: str, body: str) -> AITFeatures:
        c_h = self.compressor.compress_bits(headline)
        c_b = self.compressor.compress_bits(body)
        c_w_h = self.world_model.compute_generation_complexity(headline)
        u_h = max(0.0, c_w_h - c_h)
        ncd_hb = self.compressor.ncd(headline, body)

        # Pure AIT Engagement / Clickbait Index:
        # Unexpectedness density modulated by semantic discrepancy (NCD)
        ait_score = (u_h / (c_h + 1e-5)) * ncd_hb

        return AITFeatures(
            c_desc_headline=c_h,
            c_world_headline=c_w_h,
            unexpectedness_h=u_h,
            ncd_headline_body=ncd_hb,
            c_desc_body=c_b,
            ait_score=ait_score
        )

    def featurize_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        records = []
        for _, row in df.iterrows():
            f = self.extract_features(str(row["headline"]), str(row["body"]))
            records.append({
                "headline": row["headline"],
                "C(H)": f.c_desc_headline,
                "C_w(H)": f.c_world_headline,
                "U(H)": f.unexpectedness_h,
                "NCD(H,B)": f.ncd_headline_body,
                "AIT_Score": f.ait_score,
                "label": int(row["label"])
            })
        return pd.DataFrame(records)


class PureAITClassifier:
    """
    Non-parametric classifier: predicts clickbait solely via optimal MDL threshold
    calibrated on the AIT Score = (U(H) / C(H)) * NCD(H, B).
    """
    def __init__(self):
        self.threshold = 0.5

    def fit(self, scores: np.ndarray, y: np.ndarray):
        # Finds threshold maximizing accuracy on training data
        thresholds = np.linspace(scores.min(), scores.max(), 100)
        best_acc, best_t = 0.0, thresholds[0]
        for t in thresholds:
            acc = accuracy_score(y, scores >= t)
            if acc > best_acc:
                best_acc = acc
                best_t = t
        self.threshold = best_t

    def predict(self, scores: np.ndarray) -> np.ndarray:
        return (scores >= self.threshold).astype(int)


# ============================================================================
# 3. Dataset Helpers (Synthetic + Real File Loader)
# ============================================================================

def load_clickbait_dataset(csv_path: str = None) -> pd.DataFrame:
    """
    Loads dataset from CSV if available, otherwise returns the synthetic benchmark.
    Expected CSV columns: 'headline', 'body', 'label' (0 or 1).
    """
    if csv_path:
        df = pd.read_csv(csv_path)
        required = {"headline", "body", "label"}
        if not required.issubset(df.columns):
            raise ValueError(f"CSV must contain columns: {required}")
        return df

    # Benchmark incorporato
    return pd.DataFrame([
        {"headline": "You Won't Believe What This Animal Did in Marlow!", "body": "A resident in Marlow opened a kitchen drawer to find a corn snake. RSPCA was called.", "label": 1},
        {"headline": "This 9-Year Old Stunned Scientists With One Simple Number Trick", "body": "A student noticed a symmetry in the multiplication table of nine during regular school activities.", "label": 1},
        {"headline": "South Africa Lottery Probed as Consecutive Numbers 5, 6, 7, 8, 9, 10 Drawn!", "body": "The lottery operator confirmed the sequence was drawn legitimately, paying out 20 winners.", "label": 1},
        {"headline": "Secret Diet Trick Doctors Do Not Want You To Discover", "body": "Drinking water before meals helps moderate caloric intake, according to nutritional guidelines.", "label": 1},
        {"headline": "Corn Snake Found in Kitchen Drawer in Marlow Removed by RSPCA", "body": "A non-venomous corn snake was safely recovered from a kitchen drawer in Marlow by officers on Friday.", "label": 0},
        {"headline": "European Central Bank Raises Key Interest Rate by 25 Basis Points", "body": "The European Central Bank announced an increase of 25 basis points in interest rates.", "label": 0},
        {"headline": "French Lottery Results for Tuesday October 2026", "body": "The winning numbers for Tuesday's national lottery were drawn: 12, 19, 28, 33, 41 and bonus ball 4.", "label": 0},
        {"headline": "Greenland Peat Wildfires Extinguished After Heavy Rainfall", "body": "Wildfires that had burned through peat layers in western Greenland were extinguished after rain.", "label": 0}
    ])


# ============================================================================
# 4. Main Execution
# ============================================================================

def main():
    print("[*] Caricamento Modello Mondiale di Generazione (GPT-2)...")
    world_model = WorldModelComplexity(model_name="gpt2")
    analyzer = NewsAITAnalyzer(world_model=world_model)

    df_raw = load_clickbait_dataset()
    print(f"[*] Elaborazione di {len(df_raw)} articoli...")
    df_metrics = analyzer.featurize_dataframe(df_raw)

    print("\n" + "=" * 70)
    print(" METRICHE TEORICHE AIT PER ARTICOLO")
    print("=" * 70)
    print(df_metrics[["headline", "U(H)", "NCD(H,B)", "AIT_Score", "label"]].to_string())

    # Valutazione del Classificatore Puramente AIT con Cross-Validation
    scores = df_metrics["AIT_Score"].values
    y = df_metrics["label"].values

    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    all_preds = np.zeros_like(y)

    for train_idx, test_idx in skf.split(scores, y):
        ait_clf = PureAITClassifier()
        ait_clf.fit(scores[train_idx], y[train_idx])
        all_preds[test_idx] = ait_clf.predict(scores[test_idx])

    acc = accuracy_score(y, all_preds)
    auc = roc_auc_score(y, scores)

    print("\n" + "=" * 70)
    print(" RISULTATI DEL CLASSIFICATORE BASATO PURAMENTE SU AIT")
    print("=" * 70)
    print(f"Accuracy AIT: {acc * 100:.2f}%")
    print(f"ROC-AUC AIT : {auc:.4f}")
    print("\nReport di Classificazione:")
    print(classification_report(y, all_preds, target_names=["News Standard", "Clickbait"]))


if __name__ == "__main__":
    main()

[*] Caricamento Modello Mondiale di Generazione (GPT-2)...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (4 outputs left out)

[*] Elaborazione di 8 articoli...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



 METRICHE TEORICHE AIT PER ARTICOLO
                                                                      headline  U(H)  NCD(H,B)  AIT_Score  label
0                            You Won't Believe What This Animal Did in Marlow!   0.0  0.527778        0.0      1
1              This 9-Year Old Stunned Scientists With One Simple Number Trick   0.0  0.578947        0.0      1
2  South Africa Lottery Probed as Consecutive Numbers 5, 6, 7, 8, 9, 10 Drawn!   0.0  0.540541        0.0      1
3                        Secret Diet Trick Doctors Do Not Want You To Discover   0.0  0.567568        0.0      1
4                Corn Snake Found in Kitchen Drawer in Marlow Removed by RSPCA   0.0  0.461538        0.0      0
5            European Central Bank Raises Key Interest Rate by 25 Basis Points   0.0  0.333333        0.0      0
6                              French Lottery Results for Tuesday October 2026   0.0  0.538462        0.0      0
7                   Greenland Peat Wildfires Extinguished A

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
